# Training CNN models, implementing CBAM and comparing base models vs hybrid models.

In [ ]:
# Core libraries
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

# TensorFlow / Keras

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, backend as K, Model
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, Callback
from tensorflow.keras.utils import Sequence, to_categorical
from tensorflow.keras.saving import register_keras_serializable
# Keras Applications

from tensorflow.keras.applications import (
    EfficientNetB0, EfficientNetB3, EfficientNetB5,
    ResNet50, ResNet101, VGG16, VGG19, InceptionV3,
    DenseNet121, MobileNetV2, MobileNetV3Large
)
from tensorflow.keras.applications.efficientnet import preprocess_input as preprocess_effnet
from tensorflow.keras.applications.resnet import preprocess_input as preprocess_resnet
from tensorflow.keras.applications.vgg16 import preprocess_input as preprocess_vgg16   
from tensorflow.keras.applications.inception_v3 import preprocess_input as preprocess_inception
from tensorflow.keras.applications.densenet import preprocess_input as preprocess_densenet
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mobilenetv2
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as preprocess_mobilenetv3
from sklearn.model_selection import train_test_split, precision_recall_fscore_support

# Data Augmentation

import albumentations as A

# Scikit-learn metrics & utilities

from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score,
    accuracy_score, cohen_kappa_score
)
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# PATHS: Using just the local data for training, validation and testing.

TRAIN_PATH = Path(r"../data/dataset/train")
VAL_PATH = Path(r"../data/dataset/val")
TEST_PATH = Path(r"../data/dataset/test")
RESULTS_PATH = Path("../models")
RESULTS_PATH.mkdir(exist_ok=True)

TRAIN_CLASSES = sorted([d.name for d in TEST_PATH.iterdir() if d.is_dir()])
class_indices = {cls: i for i, cls in enumerate(TRAIN_CLASSES)}

CONF_MATRIX_DIR = RESULTS_PATH / "confusion_matrix"
CONF_MATRIX_DIR.mkdir(parents=True, exist_ok=True)

# If want to use the public images corresponding to common classes for both local and public (for the Transfer Learning experiment):

#TRAIN_PATH  = Path(r"../data/EXPERIMENT1/domain_adaptation/train")
#VAL_PATH  = Path(r"../data/EXPERIMENT1/domain_adaptation/val")

# PARAMS FOR TRANING
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30

VAL_SPLIT = 0.2
SEED = 42

# Custom augment:

CUSTOM_AUGMENTATION = A.Compose([
    A.Rotate(limit=10, border_mode=cv2.BORDER_CONSTANT, fill=0, p=0.5),
    A.RandomResizedCrop(
        size=(IMG_SIZE[0], IMG_SIZE[1]),
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        interpolation=cv2.INTER_LINEAR,
        p=0.3
    ),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),

    A.RandomBrightnessContrast(brightness_limit=0.08, contrast_limit=0.08, p=0.3),

    A.HueSaturationValue(hue_shift_limit=3, sat_shift_limit=5, val_shift_limit=4, p=0.2),

    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.05,
        rotate_limit=0,
        border_mode=cv2.BORDER_CONSTANT,
        fill=0,
        p=0.3
    ),

    A.Resize(width=IMG_SIZE[1], height=IMG_SIZE[0])
], p=1.0)

Custom data generator for using data custom data augmentation and preprocessing. 

In [ ]:
class AlbumentationsDataGenerator(Sequence):
    """
    Data generator using Albumentations. Allows loading from a folder or directly
    from a provided list of (path, label) samples.
    """

    def __init__(self, folder_path=None, samples_list=None,
                 batch_size=32, img_size=(224, 224),
                 shuffle=True, augment=None, min_per_class=0,
                 mean=None, std=None, oversample=False,
                 backbone="efficientnetb0", return_filenames=False):

        self.folder_path = Path(folder_path) if folder_path else None
        self.batch_size = batch_size
        self.img_size = img_size
        self.shuffle = shuffle
        self.augment = augment
        self.oversample = oversample
        self.min_per_class = min_per_class
        self.mean = mean
        self.std = std
        self.backbone = backbone.lower()
        self.return_filenames = return_filenames

        # if sample list provided, then use it
        if samples_list is not None:
            self.samples = samples_list
            # obtain classes from labels
            labels = [label for _, label in self.samples]
            unique_labels = sorted(set(labels))
            self.class_indices = {cls: i for i, cls in enumerate(unique_labels)}
            self.classes = unique_labels

        else:
            # load from folder
            self.classes = sorted([d.name for d in self.folder_path.iterdir() if d.is_dir()])
            self.class_indices = {cls: i for i, cls in enumerate(self.classes)}
            self.samples = []

            exts = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]
            class_samples = {
                cls: sum([list((self.folder_path / cls).glob(e)) for e in exts], [])
                for cls in self.classes
            }

            for cls in self.classes:
                imgs = class_samples.get(cls, [])
                if len(imgs) == 0:
                    print(f"[WARNING] Clase '{cls}' sin imágenes")
                    continue
                if self.oversample:
                    n_repeat = max(1, int(np.ceil(self.min_per_class / len(imgs))))
                    self.samples += [(p, self.class_indices[cls]) for p in imgs for _ in range(n_repeat)]
                else:
                    self.samples += [(p, self.class_indices[cls]) for p in imgs]

        self.on_epoch_end()

    def __len__(self):
        return max(1, int(np.ceil(len(self.samples) / self.batch_size)))

    def __getitem__(self, index):
        batch_samples = self.samples[index * self.batch_size:(index + 1) * self.batch_size]
        X, y, filenames = [], [], []

        for img_path, label in batch_samples:
            try:
                image = cv2.imread(str(img_path))
                if image is None:
                    continue
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                image = cv2.resize(image, self.img_size)
                image = image.astype(np.float32) / 255.0

                if self.mean is not None and self.std is not None:
                    image = (image - self.mean) / self.std

                if self.augment:
                    augmented = self.augment(image=image)
                    image = augmented['image']

                X.append(image)
                y.append(label)
                filenames.append(str(img_path))
            except Exception as e:
                print(f"[ERROR] {img_path} -> {e}")

        if len(X) == 0:
            empty_X = np.zeros((0, *self.img_size, 3), dtype=np.float32)
            empty_y = np.zeros((0, len(self.classes)), dtype=np.float32)
            return (empty_X, empty_y, []) if self.return_filenames else (empty_X, empty_y)

        X = np.stack(X, axis=0).astype(np.float32)
        X = self._preprocess_backbone(X)
        y = to_categorical(y, num_classes=len(self.classes))

        if self.return_filenames:
            return X, y, filenames
        else:
            return X, y

    def on_epoch_end(self):
        if self.shuffle:
            random.shuffle(self.samples)

    def get_labels(self):
        return [label for _, label in self.samples]

    def _preprocess_backbone(self, X):
        b = self.backbone
        X255 = X * 255.0

        if "resnet" in b:
            return preprocess_resnet(X255)
        elif "vgg" in b:
            return preprocess_vgg16(X255)
        elif "inception" in b:
            return preprocess_inception(X255)
        elif "efficientnet" in b:
            return preprocess_effnet(X255)
        elif "densenet" in b:
            return preprocess_densenet(X255)
        elif "mobilenetv2" in b:
            return preprocess_mobilenetv2(X255)
        elif "mobilenetv3" in b:
            return preprocess_mobilenetv3(X255)
        else:
            return X255

In [ ]:
class ClassMetricsCallback(Callback):
    """
    Callback to compute metrics for each class on validation data
    """

    def __init__(self, val_gen, class_names):
        """
        val_gen: validation data generator created using previous AlbumentationsDataGenerator
        class_names: list of class names
        """
        super().__init__()
        self.val_gen = val_gen
        self.class_names = class_names

    def on_epoch_end(self, epoch, logs=None):
        """
        Computes metrics for each class on validation data at the end of each epoch.
        The metrics considered are: precision, recall and f1 but averaged as macro,
        in order to balance the impact of different classes.
        """
        y_true, y_pred = [], []

        for i in range(len(self.val_gen)):
            X_val, y_val = self.val_gen[i]
            if X_val.shape[0] == 0:
                continue
            preds = self.model.predict(X_val, verbose=0)
            y_true.extend(np.argmax(y_val, axis=1))
            y_pred.extend(np.argmax(preds, axis=1))

        if len(y_true) > 0 and len(y_pred) > 0:
            precisions = precision_score(y_true, y_pred, average=None, zero_division=0)
            recalls = recall_score(y_true, y_pred, average=None, zero_division=0)
            f1s = f1_score(y_true, y_pred, average=None, zero_division=0)

            print(f"\n[CLASS METRICS]")
            for i, cls in enumerate(self.class_names):
                print(f"{cls}: Precision={precisions[i]:.3f}, Recall={recalls[i]:.3f}, F1={f1s[i]:.3f}")

            logs['val_precision_macro'] = precision_score(y_true, y_pred, average='macro', zero_division=0)
            logs['val_recall_macro'] = recall_score(y_true, y_pred, average='macro', zero_division=0)
            logs['val_f1_macro'] = f1_score(y_true, y_pred, average='macro', zero_division=0)

            print(
                f" — val_precision_macro: {logs['val_precision_macro']:.4f}"
                f" — val_recall_macro: {logs['val_recall_macro']:.4f}"
                f" — val_f1_macro: {logs['val_f1_macro']:.4f}\n"
            )
        else:
            print("validation metrics not computed (empty batch)")


# customized loss:
@tf.keras.utils.register_keras_serializable()
def weighted_categorical_crossentropy(weights):
    """
    Cross-entropy loss weighted using class imbalance: minority class has greater weight.
    weights: array de pesos (1D) con un valor por clase.
    """
    weights = tf.constant(weights, dtype=tf.float32)

    def loss(y_true, y_pred):
        unweighted_loss = tf.keras.losses.categorical_crossentropy(y_true, y_pred)
        weight_per_sample = tf.reduce_sum(weights * y_true, axis=-1)
        return unweighted_loss * weight_per_sample

    return loss

In [ ]:
def recall_macro(y_true, y_pred, NUM_CLASSES=6):
    y_true = K.argmax(y_true, axis=-1)
    y_pred = K.argmax(y_pred, axis=-1)

    recalls = []
    for i in range(NUM_CLASSES):
        y_true_i = K.cast(K.equal(y_true, i), 'float32')
        y_pred_i = K.cast(K.equal(y_pred, i), 'float32')
        tp = K.sum(y_true_i * y_pred_i)
        fn = K.sum(y_true_i * (1 - y_pred_i))
        recall = tp / (tp + fn + K.epsilon())
        recalls.append(recall)
    return K.mean(tf.stack(recalls))


def precision_macro(y_true, y_pred, NUM_CLASSES=6):
    y_true = K.argmax(y_true, axis=-1)
    y_pred = K.argmax(y_pred, axis=-1)

    precisions = []
    for i in range(NUM_CLASSES):
        y_true_i = K.cast(K.equal(y_true, i), 'float32')
        y_pred_i = K.cast(K.equal(y_pred, i), 'float32')
        tp = K.sum(y_true_i * y_pred_i)
        fp = K.sum((1 - y_true_i) * y_pred_i)
        precision = tp / (tp + fp + K.epsilon())
        precisions.append(precision)
    return K.mean(tf.stack(precisions))


def f1_macro(y_true, y_pred, NUM_CLASSES=6):
    p = precision_macro(y_true, y_pred, NUM_CLASSES)
    r = recall_macro(y_true, y_pred, NUM_CLASSES)
    return 2 * (p * r) / (p + r + K.epsilon())

In [ ]:

# backbone -> model from keras.
BACKBONES = {
    "resnet50": ResNet50,
    "resnet101": ResNet101,
    "vgg16": VGG16,
    "vgg19": VGG19,
    "inceptionv3": InceptionV3,
    "efficientnetb0": EfficientNetB0,
    "efficientnetb3": EfficientNetB3,
    "efficientnetb5": EfficientNetB5,
    "densenet121": DenseNet121,
    "mobilenetv2": MobileNetV2,
    "mobilenetv3large": MobileNetV3Large
}


def build_model(backbone_name, input_shape=(224,224,3), num_classes=6, weights="imagenet"):
    """
    backbone_name: str ("resnet50", "efficientnetb0", etc.)
    input_shape: tuple (H, W, 3)
    num_classes: number of output classes
    weights: name of the dataset of pre-learned weights for the backbone.
    """
    backbone_name = backbone_name.lower()
    if backbone_name not in BACKBONES:
        raise ValueError(f"Backbone {backbone_name} no soportado. Opciones: {list(BACKBONES.keys())}")

    BackboneClass = BACKBONES[backbone_name]

    # Backbone
    base_model = BackboneClass(
        include_top=False,
        input_shape=input_shape,
        weights=weights,
        pooling=None
    )

    # head modified
    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs=base_model.input, outputs=output)

    return model, base_model


In [ ]:
model_list = [
    "mobilenetv2",
    "efficientnetb0",
    "mobilenetv3large",
    "densenet121",
    "efficientnetb3",
    "resnet50",
    "Inceptionv3",
    "resnet101",
    "vgg16",
    "vgg19",
    "efficientnetb5"
]

## MODIFICAR ESTA FUNCIÓN SI SE QUIERE HACER K-FOLD. ACTUALMENTE DIVIDE EN TRAIN / TEST / VAL DE FORMA FIJA:

In [ ]:

def split_dataset_three_way(dataset_path, seed=42):
    """
    Divide las imágenes del dataset público en tres subconjuntos estratificados:
    64% train, 16% val y 20% test, manteniendo la estructura de carpetas.
    """

    base_dir = dataset_path.parent / "domain_adaptation"
    train_dir = base_dir / "train_public_64"
    val_dir   = base_dir / "val_public_16"
    test_dir  = base_dir / "test_public_20"

    for d in [train_dir, val_dir, test_dir]:
        d.mkdir(parents=True, exist_ok=True)

    random.seed(seed)

    for cls in os.listdir(dataset_path):
        cls_path = dataset_path / cls
        if not cls_path.is_dir():
            continue

        images = list(cls_path.glob("*.*"))
        if len(images) == 0:
            continue


        trainval_imgs, test_imgs = train_test_split(
            images, test_size=0.20, random_state=seed, stratify=None
        )


        train_imgs, val_imgs = train_test_split(
            trainval_imgs, test_size=0.20, random_state=seed
        )


        for subset_dir, subset_imgs in zip(
            [train_dir, val_dir, test_dir], [train_imgs, val_imgs, test_imgs]
        ):
            (subset_dir / cls).mkdir(parents=True, exist_ok=True)
            for img in subset_imgs:
                dst = subset_dir / cls / img.name
                if not dst.exists():
                    os.system(f'cp "{img}" "{dst}"')

    print(f"✅ Dataset dividido correctamente en:\n"
          f" - Train (64%): {train_dir}\n"
          f" - Val (16%):   {val_dir}\n"
          f" - Test (20%):  {test_dir}")

    return train_dir, val_dir, test_dir


## Fine-tuning pre-trained models

In [ ]:
def preprocess_for_backbone(img, backbone_name):
    """
    Apply the appropriate preprocessing to the image according to the model architecture.
    """
    X255 = img * 255.0
    b = backbone_name.lower()

    if "resnet" in b:
        return preprocess_resnet(X255)
    elif "vgg" in b:
        return preprocess_vgg16(X255)
    elif "inception" in b:
        return preprocess_inception(X255)
    elif "efficientnet" in b:
        return preprocess_effnet(X255)
    elif "densenet" in b:
        return preprocess_densenet(X255)
    elif "mobilenetv2" in b:
        return preprocess_mobilenetv2(X255)
    elif "mobilenetv3" in b:
        return preprocess_mobilenetv3(X255)
    else:
        return img

In [ ]:
for backbone_name in model_list:
    print(f"\n=== Training model: {backbone_name} ===")

    # data gen
    train_gen = AlbumentationsDataGenerator(
        folder_path=TRAIN_PATH,
        batch_size=BATCH_SIZE,
        img_size=IMG_SIZE,
        augment=CUSTOM_AUGMENTATION,
        oversample=True,
        min_per_class=400,
        backbone=backbone_name
    )

    val_gen = AlbumentationsDataGenerator(
        folder_path=VAL_PATH,
        batch_size=BATCH_SIZE,
        img_size=IMG_SIZE,
        shuffle=False,
        augment=None,
        oversample=False,
        backbone=backbone_name
    )

    # weights
    NUM_CLASSES = len(train_gen.classes)
    y_train = train_gen.get_labels()
    weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
    weights = np.sqrt(weights)
    # Model
    model, backbone = build_model(backbone_name, input_shape=IMG_SIZE + (3,), num_classes=NUM_CLASSES)

    # Callbacks
    best_model_path = str(RESULTS_PATH / f"{backbone_name}_best_model.keras")
    callbacks = [
        ModelCheckpoint(best_model_path, save_best_only=True, monitor='val_f1_macro', mode='max'),
        EarlyStopping(monitor='val_f1_macro', patience=5, restore_best_weights=True, mode='max'),
        ReduceLROnPlateau(monitor='val_f1_macro', factor=0.5, patience=2, min_lr=1e-6, mode='max'),
        ClassMetricsCallback(val_gen, class_names=train_gen.classes)
    ]

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss=weighted_categorical_crossentropy(weights),
        metrics=['accuracy', precision_macro, recall_macro, f1_macro]
    )

    # Train
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=30,
        callbacks=callbacks
    )

    print(f"\n=== TRAINING COMPLETE FOR {backbone_name} ===\n")

In [ ]:
for backbone_name in model_list:
    print(f"\n=== EVALUATING MODEL: {backbone_name} ===\n")

    best_model_path = RESULTS_PATH / f"{backbone_name}_best_model.keras"
    if not best_model_path.exists():
        print(f"Model not found: {backbone_name}")
        continue

    model = tf.keras.models.load_model(best_model_path, compile=False)

    X_list, y_list = [], []

    for cls in TRAIN_CLASSES:
        class_dir = TEST_PATH / cls
        if not class_dir.exists():
            print(f"Absent class: {cls}")
            continue

        imgs = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png")) + list(class_dir.glob("*.jpeg"))

        for img_path in imgs:
            img = cv2.imread(str(img_path))
            if img is None:
                continue

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, IMG_SIZE)

            img = img.astype(np.float32) / 255.0
            img = preprocess_for_backbone(img, backbone_name) 

            X_list.append(img)
            y_list.append(class_indices[cls])

    if not X_list:
        print(f"Not available images for backbone: {backbone_name}.")
        continue

    X_test = np.stack(X_list, axis=0)
    y_true = np.array(y_list)

    # Preds
    preds = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
    y_pred = np.argmax(preds, axis=1)

    # metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    kappa = cohen_kappa_score(y_true, y_pred)

    print(f"\n=== GLOBAL METRICS FOR {backbone_name} ===")
    print(f"Accuracy:       {accuracy:.4f}")
    print(f"Precision:      {precision:.4f}")
    print(f"Recall:         {recall:.4f}")
    print(f"F1-score:       {f1:.4f}")
    print(f"Cohen's Kappa:  {kappa:.4f}\n")

    # design confusion matrix
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(len(TRAIN_CLASSES))
    )

    cm_color = cm.astype(np.float32) / cm.sum(axis=1, keepdims=True)
    cm_color = np.nan_to_num(cm_color)


    plt.figure(figsize=(8, 6)) 

    sns.heatmap(
        cm_color,     
        annot=cm,    
        fmt="d",
        cmap="Blues",
        xticklabels=TRAIN_CLASSES,
        yticklabels=TRAIN_CLASSES,
        vmin=0, vmax=1
    )

    plt.xlabel("Predicted")
    plt.ylabel("Real")

    plt.xticks(rotation=45, ha="right")
    plt.tight_layout() 

    out_path = CONF_MATRIX_DIR / f"confusion_matrix_{backbone_name}.png"
    plt.savefig(out_path, dpi=300)


    plt.show() 
    plt.close()

    print(f"Saved confusion matrix in: {out_path}")

# CBAM MODELS: CREATION AND TRAINING

In [ ]:
@register_keras_serializable(package="Custom")
class SpatialPooling(layers.Layer):
    def __init__(self, **kwargs):
        super(SpatialPooling, self).__init__(**kwargs)

    def call(self, inputs):
        avg_pool = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(inputs, axis=-1, keepdims=True)
        return tf.concat([avg_pool, max_pool], axis=-1)

    def get_config(self):
        config = super(SpatialPooling, self).get_config()
        return config


# CBAM BLOCK

def cbam_block(input_tensor, reduction_ratio=16):
    channel = input_tensor.shape[-1]

    # Channel Attention (original paper)
    shared_dense_one = layers.Dense(channel // reduction_ratio,
                                    activation='relu',
                                    kernel_initializer='he_normal',
                                    use_bias=True)
    shared_dense_two = layers.Dense(channel,
                                    kernel_initializer='he_normal',
                                    use_bias=True)

    avg_pool = layers.GlobalAveragePooling2D()(input_tensor)
    avg_pool = layers.Reshape((1, 1, channel))(avg_pool)
    avg_pool = shared_dense_one(avg_pool)
    avg_pool = shared_dense_two(avg_pool)

    max_pool = layers.GlobalMaxPooling2D()(input_tensor)
    max_pool = layers.Reshape((1, 1, channel))(max_pool)
    max_pool = shared_dense_one(max_pool)
    max_pool = shared_dense_two(max_pool)

    channel_attention = layers.Add()([avg_pool, max_pool])
    channel_attention = layers.Activation('sigmoid')(channel_attention)
    channel_refined = layers.Multiply()([input_tensor, channel_attention])

    # Spatial Attention
    concat = SpatialPooling()(channel_refined)
    spatial_attention = layers.Conv2D(filters=1, kernel_size=7, strides=1, padding='same',
                                      activation='sigmoid', kernel_initializer='he_normal', use_bias=False)(concat)

    refined = layers.Multiply()([channel_refined, spatial_attention])
    return refined

def build_backbone_cbam(model_name="ResNet101",
                        input_shape=(224,224,3),
                        num_classes=6,
                        weights="imagenet",
                        reduction_ratio=16,
                        dropout_rate=0.5):

    Backbone = getattr(tf.keras.applications, model_name)

    inp = layers.Input(shape=input_shape)

    backbone = Backbone(include_top=False,
                        weights=weights,
                        input_tensor=inp)

    # CBAM before classifier
    feat = cbam_block(backbone.output, reduction_ratio=reduction_ratio)

    # final head
    feat = layers.GlobalAveragePooling2D()(feat)
    feat = layers.Flatten()(feat)
    feat = layers.Dense(256, activation='relu')(feat)
    feat = layers.Dropout(dropout_rate)(feat)
    out = layers.Dense(num_classes, activation='softmax')(feat)

    model = Model(inputs=inp, outputs=out,
                  name=f"{model_name}_CBAM")

    return model, backbone

In [ ]:
experiments = [
    {"name": "efficientnetb0", "backbone": "efficientnetb0", "use_cbam": False},
    {"name": "EffNetB0_CBAM", "backbone": "efficientnetb0", "use_cbam": True},
    {"name": "mobilenetv2",    "backbone": "mobilenetv2",    "use_cbam": False},
    {"name": "MobileNetV2_CBAM",    "backbone": "mobilenetv2",    "use_cbam": True},
]

In [ ]:

for exp in experiments:
    print(f"\n=== Ejecutando: {exp['name']} ===")

    # data gen
    train_gen = AlbumentationsDataGenerator(
        folder_path=TRAIN_PATH,
        batch_size=BATCH_SIZE,
        img_size=IMG_SIZE,
        augment=CUSTOM_AUGMENTATION,
        oversample=True,
        min_per_class=400,
        backbone=exp['backbone']
    )

    val_gen = AlbumentationsDataGenerator(
        folder_path=VAL_PATH,
        batch_size=BATCH_SIZE,
        img_size=IMG_SIZE,
        shuffle=False,
        augment=None,
        oversample=False,
        backbone=exp['backbone']
    )
    NUM_CLASSES = len(train_gen.classes)

    y_train = train_gen.get_labels()
    weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
    weights = np.sqrt(weights)

    # model for cbam case:
    if exp['use_cbam']:
        model, backbone = build_backbone_cbam(
        input_shape=IMG_SIZE+(3,),
        num_classes=NUM_CLASSES,
        weights='imagenet'
      )



    else:
      model, backbone = build_model(exp['backbone'], input_shape=IMG_SIZE + (3,), num_classes=NUM_CLASSES)

    best_model_path = str(RESULTS_PATH / f"{exp['name']}_best_model.keras")
    callbacks = [
        ModelCheckpoint(best_model_path, save_best_only=True, monitor='val_f1_macro', mode='max'),
        EarlyStopping(monitor='val_f1_macro', patience=5, restore_best_weights=True, mode='max'),
        ReduceLROnPlateau(monitor='val_f1_macro', factor=0.5, patience=2, min_lr=1e-6, mode='max'),
        ClassMetricsCallback(val_gen, class_names=train_gen.classes)
    ]

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss=weighted_categorical_crossentropy(weights),
        metrics=['accuracy', precision_macro, recall_macro, f1_macro]
    )

    # Train
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=30,
        callbacks=callbacks
    )

# A PARTIR DE AQUI, ESTO IRIA A RESULTS

In [ ]:
resumen_final = []

for exp in experiments:
    print(f"\n" + "="*50)
    print(f"Evaluating: {exp['name']}")
    print("="*50)

    best_model_path = RESULTS_PATH / f"{exp['name']}_best_model.keras"
    if not best_model_path.exists():
        print(f"Not found: {best_model_path.name}")
        continue

    custom_objects = {
        'SpatialPooling': SpatialPooling,
        'f1_macro': f1_macro,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro
    }

    model = tf.keras.models.load_model(best_model_path, custom_objects=custom_objects, compile=False)

    X_list, y_list = [], []

    for cls in TRAIN_CLASSES:
        class_dir = TEST_PATH / cls
        if not class_dir.exists():
            continue

        imgs = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png"))
        for img_path in imgs:
            img = cv2.imread(str(img_path))
            if img is None: continue

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, IMG_SIZE)
            img = img.astype(np.float32) / 255.0 
            img = preprocess_for_backbone(img, exp['backbone'])

            X_list.append(img)
            y_list.append(class_indices[cls])

    if not X_list:
        print(f"Test images not found for {exp['name']}")
        continue

    X_test = np.stack(X_list, axis=0)
    y_true = np.array(y_list)

    # preds
    preds = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
    y_pred = np.argmax(preds, axis=1)

    # metrics
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    kappa = cohen_kappa_score(y_true, y_pred)

    print(f"\n Results for {exp['name']}:")
    print(f"Acc: {acc:.4f} | F1-weighted: {f1:.4f} | Kappa: {kappa:.4f}")

    resumen_final.append({
        'Modelo': exp['name'],
        'Backbone': exp['backbone'],
        'CBAM': exp['use_cbam'],
        'Accuracy': acc,
        'F1_Weighted': f1,
        'Kappa': kappa
    })

    # conf matrix
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(TRAIN_CLASSES)))
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=TRAIN_CLASSES, yticklabels=TRAIN_CLASSES)
    plt.xlabel("Predicted")
    plt.ylabel("Real")
    plt.title(f"Confusion Matrix: {exp['name']}")

    out_path = CONF_MATRIX_DIR / f"cm_{exp['name']}.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()


df_res = pd.DataFrame(resumen_final)


function to create the plot for the paper: (esto igual es para el results.ipynb)

In [ ]:
colors = ["#1f77b4", "#ff7f0e"]

def plot_model_metrics(df, model_name, classes):
    sub = df[df["Model"] == model_name]
    base = sub[sub["Variant"]=="base"].iloc[0]
    cbam = sub[sub["Variant"]=="cbam"].iloc[0]

    f1_base = np.array(base["F1_per_class"], dtype=float)
    f1_cbam = np.array(cbam["F1_per_class"], dtype=float)
    prec_base = np.array(base["Precision_per_class"], dtype=float)
    prec_cbam = np.array(cbam["Precision_per_class"], dtype=float)
    rec_base = np.array(base["Recall_per_class"], dtype=float)
    rec_cbam = np.array(cbam["Recall_per_class"], dtype=float)

    x = np.arange(len(classes))
    width = 0.35

    fig, ax = plt.subplots(1, 3, figsize=(13,4))
    fig.subplots_adjust(wspace=0.3)

    # --- F1 ---
    ax[0].bar(x - width/2, f1_base, width, label="Base", color=colors[0])
    ax[0].bar(x + width/2, f1_cbam, width, label="CBAM", color=colors[1])
    ax[0].set_xticks(x)
    ax[0].set_xticklabels(classes, rotation=45, ha="right")
    ax[0].set_ylim(0, 1)
    ax[0].set_title("F1 Score")

    # --- Precision ---
    ax[1].bar(x - width/2, prec_base, width, label="Base", color=colors[0])
    ax[1].bar(x + width/2, prec_cbam, width, label="CBAM", color=colors[1])
    ax[1].set_xticks(x)
    ax[1].set_xticklabels(classes, rotation=45, ha="right")
    ax[1].set_ylim(0, 1)
    ax[1].set_title("Precision")

    # --- Recall ---
    ax[2].bar(x - width/2, rec_base, width, label="Base", color=colors[0])
    ax[2].bar(x + width/2, rec_cbam, width, label="CBAM", color=colors[1])
    ax[2].set_xticks(x)
    ax[2].set_xticklabels(classes, rotation=45, ha="right")
    ax[2].set_ylim(0, 1)
    ax[2].set_title("Recall")


    ax[2].legend(loc='upper right')

    sns.despine()
    plt.tight_layout()
    plt.show()

In [ ]:
resumen_detallado = []

for exp in experiments:
    print(f"\n" + "="*50)
    print(f"Evaluating: {exp['name']}")
    print("="*50)

    best_model_path = RESULTS_PATH / f"{exp['name']}_best_model.keras"
    if not best_model_path.exists():
        print(f"Model not found: {best_model_path.name}")
        continue

    custom_objects = {
        'SpatialPooling': SpatialPooling,
        'f1_macro': f1_macro,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro
    }
    model = tf.keras.models.load_model(best_model_path, custom_objects=custom_objects, compile=False)

    X_list, y_list = [], []
    for cls in TRAIN_CLASSES:
        class_dir = TEST_PATH / cls
        if not class_dir.exists(): continue

        imgs = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png"))
        for img_path in imgs:
            img = cv2.imread(str(img_path))
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, IMG_SIZE)
            img = img.astype(np.float32) / 255.0
            img = preprocess_for_backbone(img, exp['backbone'])
            X_list.append(img)
            y_list.append(class_indices[cls])

    if not X_list:
        print(f"Test images not found for {exp['name']}")
        continue

    X_test = np.stack(X_list, axis=0)
    y_true = np.array(y_list)

    # Preds
    preds = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
    y_pred = np.argmax(preds, axis=1)

    prec_pc, rec_pc, f1_pc, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=np.arange(len(TRAIN_CLASSES)), zero_division=0
    )

    resumen_detallado.append({
        'Model': exp['backbone'],     
        'Variant': 'cbam' if exp['use_cbam'] else 'base',
        'Precision_per_class': prec_pc,
        'Recall_per_class': rec_pc,
        'F1_per_class': f1_pc,
        'Accuracy': accuracy_score(y_true, y_pred),
        'F1_Weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0)
    })

    # confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(TRAIN_CLASSES)))
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=TRAIN_CLASSES, yticklabels=TRAIN_CLASSES)
    plt.xlabel("Predicted")
    plt.ylabel("Real")
    plt.title(f"Confusion Matrix: {exp['name']}")
    out_path = CONF_MATRIX_DIR / f"cm_{exp['name']}.png"
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()

df = pd.DataFrame(resumen_detallado)

print(df[["Model", "Variant", "Accuracy", "F1_Weighted"]])

for model_name in df["Model"].unique():
    print(f"\n==================\nModel: {model_name}\n==================\n")
    plot_model_metrics(df, model_name, TRAIN_CLASSES)

In [ ]:

modelos_eval = [
    {"name": "resnet101", "backbone": "resnet101", "use_cbam": False},
    {"name": "ResNet101_CBAM", "backbone": "resnet101", "use_cbam": True}
]

resultados_print = {}

for m in modelos_eval:
    print(f"Computing metrics for {m['name']}...")
    path = RESULTS_PATH / f"{m['name']}_best_model.keras"

    model = tf.keras.models.load_model(path, compile=False,
                                       custom_objects={'SpatialPooling': SpatialPooling})


    preds = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
    y_pred = np.argmax(preds, axis=1)

    # Cálculo de métricas
    resultados_print[m['label']] = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1 (Weighted)": f1_score(y_true, y_pred, average='weighted', zero_division=0),
        "Precision (W)": precision_score(y_true, y_pred, average='weighted', zero_division=0),
        "Recall (W)": recall_score(y_true, y_pred, average='weighted', zero_division=0),
        "Kappa": cohen_kappa_score(y_true, y_pred)
    }

# 2. Imprimir tabla comparativa
print("\n" + "="*60)

metricas_llaves = ["Accuracy", "F1 (Weighted)", "Precision (W)", "Recall (W)", "Kappa"]

for met in metricas_llaves:
    base_val = resultados_print["Base"][met]
    cbam_val = resultados_print["CBAM"][met]
    print(f"{met:<20} | {base_val:<15.4f} | {cbam_val:<15.4f}")

print("="*60)

# Model evaluation

## Model evaluation upon local data (unseen domain)

In [ ]:
backbones = {
    "efficientnetb0": EfficientNetB0,
    "mobilenetv2": MobileNetV2,
    "densenet121": DenseNet121,
    "resnet50": ResNet50,
    "resnet101": ResNet101,
    "efficientnetb3": EfficientNetB3,
    "vgg16": VGG16,
    "vgg19": VGG19,
    "efficientnetb5": EfficientNetB5,
    "Inceptionv3": InceptionV3,
    "mobilenetv3large": MobileNetV3Large
}

In [ ]:
# available classes (common to both datasets)
TRAIN_CLASSES = [
    "abiotic_stress",
    "chewing_insect",
    "healthy",
    "mechanical_stress",
]
class_indices = {c: i for i, c in enumerate(TRAIN_CLASSES)}

print(f"📂 Carpetas presentes localmente: {[d.name for d in TEST_PATH.iterdir() if d.is_dir()]}")
print(f"🧠 Evaluando con clases del modelo: {TRAIN_CLASSES}")

for backbone_name in backbones.keys():
    print(f"\n=== EVALUATING MODEL: {backbone_name} ===\n")

    best_model_path = RESULTS_PATH / f"{backbone_name}_best_model.keras"
    if not best_model_path.exists():
        print(f"Model not found for {backbone_name}, skipping.")
        continue

    # model
    model = tf.keras.models.load_model(best_model_path, compile=False)

    # test images
    X_list, y_list, filenames = [], [], []

    for cls in TRAIN_CLASSES:
        class_dir = TEST_PATH / cls
        if not class_dir.exists():
            print(f"Absent class: {cls}")
            continue

        imgs = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png"))
        for img_path in imgs:
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, IMG_SIZE)
            img = img.astype(np.float32) / 255.0

            img = preprocess_for_backbone(img, backbone_name)

            X_list.append(img)
            y_list.append(class_indices[cls])
            filenames.append(str(img_path))

    if not X_list:
        print("Not available test images for this backbone, skipping evaluation.")
        continue

    X_test = np.stack(X_list, axis=0)
    y_test = to_categorical(y_list, num_classes=len(TRAIN_CLASSES))

    # Preds
    preds = model.predict(X_test, verbose=0)
    y_true = np.argmax(y_test, axis=1)
    y_pred = np.argmax(preds, axis=1)

    # metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    kappa = cohen_kappa_score(y_true, y_pred)

    print(f"\n=== GLOBAL METRICS FOR {backbone_name} ===")
    print(f"Accuracy:       {accuracy:.4f}")
    print(f"Precision:      {precision:.4f}")
    print(f"Recall:         {recall:.4f}")
    print(f"F1-score:       {f1:.4f}")
    print(f"Cohen's Kappa:  {kappa:.4f}\n")

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(TRAIN_CLASSES)))
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=TRAIN_CLASSES, yticklabels=TRAIN_CLASSES,
        annot_kws={"size": 14}
    )
    plt.xlabel("Predicted", fontsize=16)
    plt.ylabel("Real", fontsize=16)
    plt.title(f"{backbone_name}", fontsize=18)
    plt.xticks(fontsize=12, rotation=45, ha="right")
    plt.yticks(fontsize=12, rotation=0)
    out_path = CONF_MATRIX_DIR / f"confusion_matrix_{backbone_name}.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Confusion matrix saved to {out_path}")



In [ ]:
MODEL_CLASSES = [
    "abiotic_stress",
    "bacterial_spot",
    "chewing_insect",
    "healthy",
    "mechanical_stress",
    "mite_presence"
]
class_indices = {c: i for i, c in enumerate(MODEL_CLASSES)}


for backbone_name in backbones.keys():
    print(f"\n=== EVALUATING MODEL: {backbone_name} ===\n")

    best_model_path = RESULTS_PATH / f"{backbone_name}_best_model.keras"
    if not best_model_path.exists():
        print(f"Model not found for {backbone_name}, skipping.")
        continue

    model = tf.keras.models.load_model(best_model_path, compile=False)

    # test images
    X_list, y_list, filenames = [], [], []

    for cls in MODEL_CLASSES:
        class_dir = TEST_PATH / cls
        if not class_dir.exists():
            print(f"Absent class: {cls}")
            continue

        imgs = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png"))
        for img_path in imgs:
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, IMG_SIZE)
            img = img.astype(np.float32) / 255.0
            img = preprocess_for_backbone(img, backbone_name)

            X_list.append(img)
            y_list.append(class_indices[cls])
            filenames.append(str(img_path))

    if not X_list:
        print("Not available test images for this backbone, skipping evaluation.")
        continue


    X_test = np.stack(X_list, axis=0)
    y_true = np.array(y_list)

    # Preds
    preds = model.predict(X_test, verbose=0)
    y_pred = np.argmax(preds, axis=1)

    # global metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    kappa = cohen_kappa_score(y_true, y_pred)

    print(f"\n=== GLOBAL METRICS FOR {backbone_name} ===")
    print(f"Accuracy:       {accuracy:.4f}")
    print(f"Precision:      {precision:.4f}")
    print(f"Recall:         {recall:.4f}")
    print(f"F1-score:       {f1:.4f}")
    print(f"Cohen's Kappa:  {kappa:.4f}\n")

    # confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(MODEL_CLASSES)))

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=MODEL_CLASSES, yticklabels=MODEL_CLASSES,
        annot_kws={"size": 14}
    )
    plt.xlabel("Predicted", fontsize=16)
    plt.ylabel("Real", fontsize=16)
    plt.title(f"{backbone_name}", fontsize=18)
    plt.xticks(fontsize=12, rotation=45, ha="right")
    plt.yticks(fontsize=12, rotation=0)

    out_path = CONF_MATRIX_DIR / f"confusion_matrix_{backbone_name}.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Confusion matrix saved to {out_path}\n")


# COMPARING MODELS: CBAM VS BASE MODELS

In [ ]:
def load_custom_model(model_path):
    return tf.keras.models.load_model(
        model_path,
        custom_objects={
            "SpatialPooling": SpatialPooling,
            "cbam_block": cbam_block
        }
    )


def load_model_eval_only(model_path):
    return tf.keras.models.load_model(
        model_path,
        compile=False,
        custom_objects={"SpatialPooling": SpatialPooling, "cbam_block": cbam_block, "loss": None}
    )

In [ ]:
# backbone -> model from keras.
backbones = {
    "resnet50": ResNet50,
    "resnet101": ResNet101,
    "vgg16": VGG16,
    "vgg19": VGG19,
    "InceptionV3": InceptionV3,
    "efficientnetb0": EfficientNetB0,
    "efficientnetb3": EfficientNetB3,
    "efficientnetb5": EfficientNetB5,
    "densenet121": DenseNet121,
    "mobilenetv2": MobileNetV2,
    "mobilenetv3large": MobileNetV3Large
}

custom_models = {
    "efficientnetb0_cbam_gap": "EffNetB0_CBAM.keras",
    "mobilenetv2_cbam_gap": "MobileNetV2_CBAM.keras",

}

# Unify
model_files = {name: f"{name}_best_model.keras" for name in backbones.keys()}
model_files.update(custom_models)

def load_model_eval_only(model_path):
    """loading .keras model for evaluation without recompilation."""
    return load_model(model_path, compile=False)



# Evaluate models

results = []

for backbone_name, model_file in model_files.items():
    print(f"\n=== EVALUATING MODEL: {backbone_name} ===\n")

    best_model_path = RESULTS_PATH / model_file
    if not best_model_path.exists():
        print(f"⚠️ Model not found in {best_model_path}, skipping.")
        continue

    try:
        model = load_model_eval_only(best_model_path)
    except Exception as e:
        print(f"⚠️ Error loading {backbone_name}: {e}")
        continue


    # Load test dataset

    X_list, y_list = [], []

    classes = sorted([d.name for d in TEST_PATH.iterdir() if d.is_dir()])
    class_indices = {c: i for i, c in enumerate(classes)}

    for cls in classes:
        imgs = list((TEST_PATH / cls).glob("*.jpg")) + list((TEST_PATH / cls).glob("*.png"))
        for img_path in imgs:
            img = cv2.imread(str(img_path))
            if img is None:
                continue

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, IMG_SIZE)
            img = img.astype(np.float32) / 255.0

            img = preprocess_for_backbone(img, backbone_name)
            X_list.append(img)
            y_list.append(class_indices[cls])

    if len(X_list) == 0:
        print("⚠️ No available test images for this backbone.")
        continue

    X_test = np.stack(X_list, axis=0)
    y_true = np.array(y_list)

    # Predictions & metrics

    preds = model.predict(X_test, verbose=0)
    y_pred = np.argmax(preds, axis=1)

    f1 = f1_score(y_true, y_pred, average="weighted")
    recall = recall_score(y_true, y_pred, average="weighted")
    precision = precision_score(y_true, y_pred, average="weighted")
    accuracy = accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)

    results.append({
        "Backbone": backbone_name,
        "F1": round(f1, 3),
        "Recall": round(recall, 3),
        "Precision": round(precision, 3),
        "Accuracy": round(accuracy, 3),
        "Kappa": round(kappa, 3)
    })


    # Confusion matrix

    cm = confusion_matrix(y_true, y_pred)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm_normalized,                  
        annot=cm,                  
        fmt="d",                        
        cmap="Blues",
        xticklabels=classes,
        yticklabels=classes,
        annot_kws={"size":14}
    )
    plt.xlabel("Predicted", fontsize=16)
    plt.ylabel("Real", fontsize=16)
    plt.title(f"{backbone_name}", fontsize=18)
    plt.xticks(fontsize=12, rotation=45, ha="right")
    plt.yticks(fontsize=12, rotation=0)
    out_path = CONF_MATRIX_DIR / f"confusion_matrix_{backbone_name}_counts.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Confusion matrix saved to {out_path}")



# Summary results
df_results = pd.DataFrame(results)
print("\n=== METRICS SUMMARY ===\n")
print(df_results)


# save results table
results_csv = RESULTS_PATH / "evaluation_summary.csv"
df_results.to_csv(results_csv, index=False)
print(f"\n📊 Results saved to {results_csv}")
